## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [1]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

C:\Users\Marcus\AppData\Local\Temp\ipykernel_13076\3609170269.py:13: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!


In [2]:
sao_URL = "https://www.polyu.edu.hk/sao/"
docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "html.parser")   # Strip the html syntax
    
    # Get the title of the page
    title = soup.title.string if soup.title else "No Title"

    # Remove script and style elements
    for unwanted in soup(["script", "style"]):
        unwanted.decompose()
    
    # Define common navigational/junk tags to remove
    unwanted_tags = ['header', 'footer', 'nav', 'aside', 'button', 'form', 'noscript']
    
    # Attempt to find main content areas
    main_content = soup.find('main') or soup.find('article') or soup.find('div', class_='content')
    
    if main_content:
        for unwanted in main_content.find_all(unwanted_tags):
            unwanted.decompose()
        text = re.sub(r"\n\n+", "\n\n", main_content.get_text()).strip()
    else:
        for unwanted in soup.find_all(unwanted_tags):
            unwanted.decompose()
        text = re.sub(r"\n\n+", "\n\n", soup.text).strip()
    
    # Post-processing to remove lines that look like menu items
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # Skip lines that are very likely menu navigation
        if re.search(r'menu|Search|Contact Us|Sitemap', line, re.I):
            continue
        cleaned_lines.append(line)
    
    text = "\n".join(cleaned_lines)

    if len(text) < 200:
        # Check if there's an actual paragraph or significant text block
        paraagraph = soup.find_all('p')
        length = sum(len(p.get_text()) for p in paraagraph)
        if length < 100:
            return f"{title}\n\n[No significant text content]"

    return f"{title}\n\n{text}"

sao_loader = RecursiveUrlLoader(
    max_depth=6,
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"News-and-Events", 
        sao_URL+"news-and-events",
        sao_URL+"Sitemap",
        sao_URL+"sitemap",
        sao_URL+"Search-Result",
        sao_URL+"search-result",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"docdrive",
        sao_URL+"-",
        # Specifically adding problematic paths if known
        sao_URL+"counselling-and-wellness-section/polyu-asian-universities-water-polo-invitational-tournament/photo-gallery/"
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    for key in unwanted_metadata:
        if key in doc.metadata:
            del doc.metadata[key]
    docs.append(doc)

https://www.polyu.edu.hk/sao/
https://www.polyu.edu.hk/sao/student-resources-and-support-section/financial-assistance/our-donors/
https://www.polyu.edu.hk/sao/student-development-section/holistic-student-development/eagle/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/wellness-centre/polyu-wellfit-app/announcements/
https://www.polyu.edu.hk/sao/student-development-section/holistic-student-development/on-campus-intercultural-development-programmes/fun-on-campus/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/sports-development/pe-programme/fitness-training-course/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/programmes-and-activities/positivity-project/positive-psychology-ambassador/
https://www.polyu.edu.hk/sao/counselling-and-wellness-section/sports-development/sports-team/sports-team-recruitment-registration/
https://www.polyu.edu.hk/sao/student-development-section/holistic-student-development/extra-curricular-enrichment-for-lifelong-le

In [3]:
idx = 76
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(docs[idx].page_content[:])
print(docs[idx].metadata.get('source'))

'''
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 272
Organising Committee | Student Affairs Office

Organising Committee
Honorary Advisor
Dr LAM Tai Fai, Council Chairman, The Hong Kong Polytechnic University
Honorary Chairman
Professor Jin-Guang TENG, President, The Hong Kong Polytechnic University
Organising Committee Chairman
Professor Ben YOUNG, Vice President (Student and Global Affairs), The Hong Kong Polytechnic University
Organising Committee Vice Chairman
Professor Albert CHAN, Dean of Students, The Hong Kong Polytechnic University
Advisors
Dr Eric TAM, Associate Dean of Students, The Hong Kong Polytechnic University
Dr Florence WU, Section Head (Counselling and Wellness), Student Affairs Office, The Hong Kong Polytechnic University
Executive Chairman
Mr Peter CHEUNG, Physical Education Officer, Counselling and Wellness Section, Student Affairs Office, The Hong Kong Polytechnic University
Executive Secretary
Mr Adrian LIU, Physical Education Officer, Counselling and Wellness Section, Stud

"\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=300,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1) if re.search(r'/([^/]+)/?$', source) else "unknown"
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"
    
    chunk.page_content = f"--- PolyU Student Affairs Office (SAO) Website ---\n\n{chunk.page_content}"

print("Number of chunks:", len(chunks))


Number of chunks: 991


In [5]:
print(chunks[10])

page_content='--- PolyU Student Affairs Office (SAO) Website ---

To nurture students’ collaborative vision to uphold and lead with an ethical outlook and a sense of responsibility, fostering shared sustainable development on the local, national, and international levels
To forge partnerships with the PolyU community, alumni, and global societal leaders and organisations to enrich student experiences, and to reinforce PolyU’s and SAO’s position in youth leadership development
EAGLE the Adventurers
EAGLE International Youth Leaders Summit
Cross-institutional Student-led Social Projects
Education Objectives
EAGLE stands for Ethics, Aspirations, Globality, Leadership, and Excellence.  It empowers students to:
Ethics
Aspirations
Globality
Leadership
Excellence
To develop strong ethical principles for guiding future life decisions and lead with integrity
To aim high and cultivate strong motivation and resilience towards goals
To embrace a global perspective and an appreciation of diversitie

### 3. Document Embedding in Chroma

In [6]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "sao_documents" if not SINGLE else "vaa_documents"

In [7]:
client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

0

In [8]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

In [9]:
for i, chunk in enumerate(chunks):
    print(f"Adding chunk {i+1}/{len(chunks)} to ChromaDB. Metadata: {chunk.metadata}\n")
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

Adding chunk 1/991 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_0'}

Adding chunk 2/991 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_1'}

Adding chunk 3/991 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_2'}

Adding chunk 4/991 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affairs Office', 'chunk_id': 'PolyU_SAO_sao_chunk_3'}

Adding chunk 5/991 to ChromaDB. Metadata: {'source': 'https://www.polyu.edu.hk/sao/', 'content_type': 'text/html; charset=utf-8', 'title': 'Home | Student Affai

### 4. Simple Testing

In [10]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=10)

for result in results:
    print("========================================================")
    print(f"Source: {result.metadata.get('info')}")
    print(f"Content: {result.page_content}...")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

C:\Users\Marcus\AppData\Local\Temp\ipykernel_13076\4078323127.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorStore = Chroma(


Source: None
Content: --- PolyU Student Affairs Office (SAO) Website ---

Students are strongly encouraged to have hall-life experience during their university years, which is both memorable and rewarding. The Student Halls of Residence do not just provide students with an accommodation, but a vibrant community with abundant opportunities for them to grow and learn.
Application for Hall Residence 2025/26 – Full-time undergraduate students
Student Type
Application Period
Ad-hoc Application for Hall Residence 2025/26
Stage (1): 27 Jan (10:00am) – 2 Feb 2026 (11:59pm)
Stage (2): 3 Feb (10:00am) - 24 Mar 2026 (11:59pm)
Inbound exchange students of Semester 2*
2 Dec 2025 (10:00am) - 9 Dec 2025 (11:59pm)
*Global Engagement Office will inform eligible inbound exchange students the application and arrangement of hall accommodation by email in due course.
Application for Summer Hall Residence 2026 – Full-time undergraduate students
Student Type
Application Period
Full-time undergraduate student